<a href="https://colab.research.google.com/github/adzetto/marine_analysis/blob/main/su-seviyesi/donemler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bozyazı — dönem dönem analiz

Aynı analiz dört ayrı pencerede **ayrı ayrı** çalıştırılır:

| Kod | Pencere | Niçin |
|---|---|---|
| `makale` | 01.07.2009 – 13.03.2018 | Yayımlanmış değerlerle karşılaştırma |
| `son5` | son 5 yıl | Hocanın asgari isteği |
| `son10` | son 10 yıl | |
| `tum` | kaydın tamamı | 2009-07 → bugün |

Her pencere ayrı bir en küçük kareler çözümü demek. Bu yüzden dönemler
sırayla işlenir ve her birinin sonucu **hemen** diske yazılır — biri
bellek yetmediği için durursa öncekiler kaybolmaz.

**Yüksek RAM çalışma zamanı gerekiyor.**  
`Çalışma zamanı → Çalışma zamanı türünü değiştir → Yüksek RAM`

In [ ]:
%cd /content
!rm -rf /content/marine_analysis
!git clone -q https://github.com/adzetto/marine_analysis.git
%cd /content/marine_analysis/su-seviyesi
!pip install -q -r requirements.txt

import psutil, os
gb = psutil.virtual_memory().total / 1e9
print(f'\ntoplam RAM : {gb:.1f} GB   CPU: {os.cpu_count()} cekirdek')
if gb < 20:
    print('UYARI: yuksek RAM calisma zamanina gecin, yoksa cozum '
          'yarida kesilir.')

In [ ]:
!apt-get -qq update && apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super > /dev/null
print('latex kuruldu')

## Dönemleri çalıştır

Tek tek çalıştırmak isterseniz: `!python -u 13_donem_kosucu.py son5`

In [ ]:
!python -u 13_donem_kosucu.py makale

In [ ]:
!python -u 13_donem_kosucu.py son5

In [ ]:
!python -u 13_donem_kosucu.py son10

In [ ]:
!python -u 13_donem_kosucu.py tum

## Dönemleri karşılaştır

Gelgit bileşenleri dönemden döneme neredeyse değişmemeli — gelgit
astronomik bir olgudur ve ay–yer–güneş hareketleri pencereye bağlı
değildir. Değişiyorsa çözümde bir sorun var demektir.

Gelgit **dışı** istatistikler ise değişir; onlar meteorolojiye bağlıdır.

In [ ]:
import pandas as pd, glob
from IPython.display import display, Image
d = pd.read_csv('tables/10_donem_karsilastirma.csv')
display(d)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

f, ax = plt.subplots(1, 3, figsize=(13, 3.6))
i = np.arange(len(d))

ax[0].bar(i, d.M2_m * 100, color='#1f4e9c')
ax[0].axhline(9.39, color='#c0392b', ls='--', lw=1.2,
              label='makale 9.39 cm')
ax[0].set_ylabel('$M_2$ genlik (cm)')
ax[0].set_title('Gelgit — döneme duyarsız olmalı')
ax[0].legend(fontsize=8)

ax[1].bar(i, d.nontidal_std_cm, color='#1f6f3f')
ax[1].axhline(10.43, color='#c0392b', ls='--', lw=1.2,
              label='makale 10.43 cm')
ax[1].set_ylabel('gelgit dışı $\\sigma$ (cm)')
ax[1].set_title('Gelgit dışı — meteorolojiye bağlı')
ax[1].legend(fontsize=8)

ax[2].bar(i, d.nontidal_aralik_cm, color='#8a6d3b')
ax[2].axhline(129, color='#c0392b', ls='--', lw=1.2,
              label='makale 129 cm')
ax[2].set_ylabel('azami aralık (cm)')
ax[2].set_title('Azami aralık — tek olaya bağlı')
ax[2].legend(fontsize=8)

for a_ in ax:
    a_.set_xticks(i); a_.set_xticklabels(d.donem)
    a_.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.show()

## Her dönemin tabloları ve figürleri

In [ ]:
for kod in d.donem:
    print('=' * 70); print(kod.upper()); print('=' * 70)
    for p in (f'tables/09_famagusta_bicimi_{kod}.csv',
              f'tables/02_gelgit_duzeyleri_{kod}.csv'):
        if os.path.exists(p):
            print(p); display(pd.read_csv(p))
    p = f'figures/01_nontidal_pdf_cdf_{kod}.png'
    if os.path.exists(p):
        display(Image(p))

## Excel ve GitHub

Jetonu Colab'ın sol panelindeki anahtar simgesinden `GITHUB_TOKEN`
adıyla saklayın; hücreye yazmayın.

In [ ]:
!python -u 10_excel_olustur.py

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
    print('jeton alindi')
except Exception as e:
    print('Jeton okunamadi:', e)

In [ ]:
!python -u 11_sonuclari_pushla.py "Donem donem analiz sonuclari"

In [ ]:
!zip -qr /content/bozyazi_donemler.zip tables figures data *.xlsx
from google.colab import files
files.download('/content/bozyazi_donemler.zip')